# 01 — Encode the biweekly panel into tokenizer latent space

**Pipeline** (`ts_SCM_ASCM`, 2026-08-19, Jun-directed): the embedding-space arm of the
SCM/ASCM panel validation. This notebook encodes every biweekly composite (60 sites ×
2 sensors × 20 periods, 2,322 images exist) into the TerraMind tokenizer's quantized
latent — the one representation used for donor selection, weight fitting, and decoding
in notebooks 02–04 of this folder.

**Data**: `Satellite/data/biweekly_datasets/` (collaborator's 14-day nanmedian composites,
read-only, as-is). This is the 60-site sample only (10 treated + 50 controls), not the
26 × 10 = 260 snapshot-era controls: 210 of those have no P01–P20 chip, so notebooks
02–04 cannot kNN-search them. Periods `before_P01..P10` = P1–P10 (2024-05-10 … 2024-09-26),
`after_P01..P10` = P11–P20 (2024-09-27 … 2025-02-13). File paths are constructed from
the metadata columns — the inventory's `output_file` strings are the collaborator's
Dropbox paths and are never used.

**File-format note**: the biweekly tifs are stored CHANNEL-FIRST (C, 101, 101) — unlike
the finals chips' (101, 101, C) — so every read transposes to (H, W, C) before the
verbatim preprocessing; an orientation + unit sanity gate (S1 VV in dB range, S2 B2 in
reflectance range, NDVI in [−1, 1]) guards the transpose on every run.

**Fixed knobs** (identical to the embed_DiD tokenizer arm): models
`terramind_v1_tokenizer_s1grd` / `..._s2l2a` (pretrained, frozen); latent = quantized
embedding (5, 14, 14) → 980-d flattened; preprocessing −Inf→NaN → per-band chip-mean
fill → (S2 ×10⁴) → bilinear 101→224 → TerraMind v1 standardization; S1 feeds VV,VH
(first 2 channels), S2 feeds B2,B3,B4,B8,B11,B12 (first 6 channels) into the 12-band
slots [1,2,3,7,10,11] with the remaining slots at the pretraining mean. NaN fill is the
chip-mean convention (Jun's decision 2026-08-19); each latent's `valid_pixel_fraction`
is carried in the index for the quality-sensitivity arm.

**Outputs**: `Satellite/data/embeddings_tok_panel/latents_biweekly.npz` (keys
`site|sensor|period_id`) + `manifest.json`; `panel_latent_index.csv` here (one row per
site × sensor × period with availability and quality).

In [1]:
# GPU pick — must run before torch initializes CUDA
import os, subprocess
if "CUDA_VISIBLE_DEVICES" not in os.environ:
    _q = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv,noheader,nounits"],
        text=True)
    _free = sorted((int(u), int(i)) for i, u in
                   (l.split(", ") for l in _q.strip().splitlines()))
    os.environ["CUDA_VISIBLE_DEVICES"] = str(_free[0][1])
print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])

import json
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
import torch
import torch.nn.functional as F

from terratorch.registry import FULL_MODEL_REGISTRY
from terratorch.models.backbones.terramind.model.terramind_register import (
    PRETRAINED_BANDS, v1_pretraining_mean, v1_pretraining_std)

ROOT = Path("/data/wang/junh/githubs/latent-synthetic-control")
BIW = ROOT / "Satellite" / "data" / "biweekly_datasets"
TS = ROOT / "Satellite" / "notebooks" / "ts_SCM_ASCM"
LATD = ROOT / "Satellite" / "data" / "embeddings_tok_panel"
LATD.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda:0")
TIMESTEPS, SEED = 50, 0
S1_BANDS = ["VV", "VH", "VV_minus_VH"]
S2_BANDS = ["B2", "B3", "B4", "B8", "B11", "B12", "NDVI", "NDWI"]
BANDS = {"sentinel1": S1_BANDS, "sentinel2": S2_BANDS}

# --- encode/prep machinery copied verbatim from embed_DiD/04 setup (2026-08-19) ---
def read_chip(path):
    a = tifffile.imread(path).astype(np.float32)
    a[~np.isfinite(a)] = np.nan
    return a


def read_chip_biweekly(path, sensor):
    """Biweekly tifs are CHANNEL-FIRST (C,101,101) — rasterio-written planar, unlike
    the finals chips' (101,101,C). Transpose to (H,W,C) so the verbatim `prep` /
    `feats_from_raw` band slicing is correct. Band order (writer's SENSOR_BANDS):
    S1 VV,VH,VV_minus_VH; S2 B2,B3,B4,B8,B11,B12,NDVI,NDWI."""
    a = read_chip(path)
    assert a.ndim == 3 and a.shape[0] == len(BANDS[sensor]), (path, a.shape)
    return np.moveaxis(a, 0, -1)


def fill_nan(x):
    for c in range(x.shape[0]):
        b = x[c]; m = np.isnan(b)
        if m.any():
            b[m] = np.nanmean(b) if not np.isnan(b).all() else 0.0
    return x

tok = {"sentinel1": FULL_MODEL_REGISTRY.build("terramind_v1_tokenizer_s1grd",
                                              pretrained=True).to(device).eval(),
       "sentinel2": FULL_MODEL_REGISTRY.build("terramind_v1_tokenizer_s2l2a",
                                              pretrained=True).to(device).eval()}
for _m in tok.values():
    for _p in _m.parameters():
        _p.requires_grad_(False)

M = {"sentinel1": np.array(v1_pretraining_mean["tok_sen1grd@224"], dtype=np.float32),
     "sentinel2": np.array(v1_pretraining_mean["tok_sen2l2a@224"], dtype=np.float32)}
SD = {"sentinel1": np.array(v1_pretraining_std["tok_sen1grd@224"], dtype=np.float32),
      "sentinel2": np.array(v1_pretraining_std["tok_sen2l2a@224"], dtype=np.float32)}
ALL12 = PRETRAINED_BANDS["untok_sen2l2a@224"]
OUR6 = ["BLUE", "GREEN", "RED", "NIR_BROAD", "SWIR_1", "SWIR_2"]  # = B2,B3,B4,B8,B11,B12
IDX12 = [ALL12.index(b) for b in OUR6]


def prep(sensor, chip):
    """chip (H,W,C) raw -> standardized tokenizer input (C_tok, 224, 224)."""
    if sensor == "sentinel1":
        x = torch.from_numpy(fill_nan(np.ascontiguousarray(
            chip[..., :2].transpose(2, 0, 1))))[None]
        x = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)
        m, s = M[sensor], SD[sensor]
        x = (x - torch.tensor(m)[None, :, None, None]) / torch.tensor(s)[None, :, None, None]
        return x[0]
    x6 = torch.from_numpy(fill_nan(np.ascontiguousarray(
        chip[..., :6].transpose(2, 0, 1))))[None] * 10_000.0
    x6 = F.interpolate(x6, size=(224, 224), mode="bilinear", align_corners=False)
    x12 = torch.zeros(1, 12, 224, 224)
    m, s = M["sentinel2"], SD["sentinel2"]
    for ci, j in enumerate(IDX12):
        x12[0, j] = (x6[0, ci] - m[j]) / s[j]
    return x12[0]


@torch.no_grad()
def encode_batch(sensor, tensors):
    q, _, _ = tok[sensor].encode(torch.stack(tensors).to(device))
    return q.cpu()

print("setup done")

CUDA_VISIBLE_DEVICES = 0


setup done


## Panel index

One row per site × sensor × period (2,400 expected). Local tif paths are constructed
from the metadata columns; existence on disk is asserted to match the inventory's
`composite_created` flag exactly — no globbing, no Dropbox path strings.

In [2]:
roster = pd.read_csv(ROOT / "Satellite" / "data" / "daily_datasets" / "selected_site_sample.csv")
assert len(roster) == 60 and roster["site_id"].is_unique
inv = pd.read_csv(BIW / "biweekly_image_quality.csv")
assert len(inv) == 2400, len(inv)

idx = inv[["site_id", "group", "sensor", "period", "period_number", "period_id",
           "period_start", "period_end", "composite_created",
           "valid_pixel_fraction"]].copy()
idx = idx.merge(roster[["site_id", "matched_treatment_site_id", "control_rank"]],
                on="site_id", how="left", validate="many_to_one")
idx["seq"] = idx["period_number"] + np.where(idx["period"] == "after", 10, 0)
idx = idx.rename(columns={"period": "half"})
idx["tif"] = [str(BIW / r.sensor / r.group / r.half /
                  f"{r.site_id}_{r.period_id}_{r.period_start}_{r.period_end}"
                  f"_biweekly_{r.sensor}.tif")
              for r in idx.itertuples()]
idx["file_exists"] = idx["tif"].map(lambda p: Path(p).exists())

# the constructed paths must reproduce the inventory's created/missing pattern exactly
assert (idx["file_exists"] == (idx["composite_created"] == 1)).all(), \
    "constructed paths do not match inventory composite_created"
idx["has_latent"] = idx["file_exists"]
n_files = int(idx["file_exists"].sum())
print(f"panel index: {len(idx)} rows, {n_files} images on disk "
      f"({len(idx) - n_files} site-periods with no acquisition)")
assert n_files == 2322, n_files
print(idx.groupby(["sensor", "half"])["file_exists"].sum())

panel index: 2400 rows, 2322 images on disk (78 site-periods with no acquisition)
sensor     half  
sentinel1  after     559
           before    588
sentinel2  after     575
           before    600
Name: file_exists, dtype: int64


## Encode

Batched encode of all 2,322 images (load-if-exists cache), then a determinism gate:
the first batch is re-encoded and must match exactly.

In [3]:
cache_f = LATD / "latents_biweekly.npz"
lat = {}
if cache_f.exists():
    _z = np.load(cache_f)
    lat = {tuple(k.split("|")): torch.from_numpy(_z[k]) for k in _z.files}
    print(f"cache loaded: {len(lat)} latents")

todo = idx.loc[idx["file_exists"]]
missing = [r for r in todo.itertuples()
           if (r.site_id, r.sensor, r.period_id) not in lat]
print(f"to encode: {len(missing)} of {len(todo)}")

# orientation + unit sanity gate on one chip per sensor (native units after transpose)
for sensor, r in [(s, next(r for r in todo.itertuples() if r.sensor == s))
                  for s in ("sentinel1", "sentinel2")]:
    c = read_chip_biweekly(r.tif, sensor)
    assert c.shape[:2] == (101, 101), c.shape
    if sensor == "sentinel1":
        vv = np.nanmean(c[..., 0])
        assert -35.0 < vv < 5.0, f"VV mean {vv} out of dB range — band order wrong?"
    else:
        b2, ndvi = np.nanmean(c[..., 0]), np.nanmean(c[..., 6])
        assert 0.0 < b2 < 0.5, f"B2 mean {b2} out of reflectance range"
        assert -1.001 <= ndvi <= 1.001, f"NDVI mean {ndvi} out of range"
    print(f"orientation gate {sensor}: shape {c.shape} OK")

BATCH = 64
for sensor in ("sentinel1", "sentinel2"):
    rows = [r for r in missing if r.sensor == sensor]
    for s0 in range(0, len(rows), BATCH):
        chunk = rows[s0:s0 + BATCH]
        tensors = [prep(sensor, read_chip_biweekly(r.tif, sensor)) for r in chunk]
        q = encode_batch(sensor, tensors)
        for r, qi in zip(chunk, q):
            lat[(r.site_id, r.sensor, r.period_id)] = qi
        print(f"{sensor}: {min(s0 + BATCH, len(rows))}/{len(rows)}", flush=True)

assert len(lat) == 2322, len(lat)
shape = lat[next(iter(lat))].shape
assert all(v.shape == shape for v in lat.values())
print("latent shape:", tuple(shape), "D =", int(np.prod(shape)))

# determinism gate — re-encode one batch, require exact match
for sensor in ("sentinel1", "sentinel2"):
    rows = [r for r in todo.itertuples() if r.sensor == sensor][:16]
    q2 = encode_batch(sensor, [prep(sensor, read_chip_biweekly(r.tif, sensor))
                               for r in rows])
    g = max(float((lat[(r.site_id, r.sensor, r.period_id)] - qi).abs().max())
            for r, qi in zip(rows, q2))
    print(f"determinism gate {sensor}: max|diff| = {g:.2e}")
    assert g < 1e-5

np.savez_compressed(cache_f, **{"|".join(k): v.numpy() for k, v in lat.items()})
print("saved:", cache_f, f"({cache_f.stat().st_size/1e6:.1f} MB)")

to encode: 2322 of 2322
orientation gate sentinel1: shape (101, 101, 3) OK
orientation gate sentinel2: shape (101, 101, 8) OK


sentinel1: 64/1147


sentinel1: 128/1147


sentinel1: 192/1147


sentinel1: 256/1147


...

sentinel2: 1175/1175


latent shape: (5, 14, 14) D = 980


determinism gate sentinel1: max|diff| = 0.00e+00


determinism gate sentinel2: max|diff| = 0.00e+00


saved: /data/wang/junh/githubs/latent-synthetic-control/Satellite/data/embeddings_tok_panel/latents_biweekly.npz (2.1 MB)


## Index + manifest

In [4]:
out_idx = idx[["site_id", "group", "matched_treatment_site_id", "control_rank",
               "sensor", "period_id", "seq", "half", "period_start", "period_end",
               "has_latent", "valid_pixel_fraction"]]
out_idx.to_csv(TS / "panel_latent_index.csv", index=False)
print("saved:", TS / "panel_latent_index.csv", out_idx.shape)

manifest = {
    "date": "2026-08-19",
    "dataset": "biweekly_datasets (60 sites x 2 sensors x 20 periods, 2322 images)",
    "models": {"sentinel1": "terramind_v1_tokenizer_s1grd",
               "sentinel2": "terramind_v1_tokenizer_s2l2a"},
    "latent": "quantized (encode -> q); flattened for vector arithmetic",
    "shape": {"sentinel1": [5, 14, 14], "sentinel2": [5, 14, 14]},
    "preprocessing": ("nodata -Inf->NaN->per-band chip mean; bilinear 101->224; "
                      "TerraMind v1 standardization; S1 VV,VH dB; S2 reflectance "
                      "x10000, 6-of-12 bands, missing at pretraining mean"),
    "decode": {"timesteps": TIMESTEPS, "seed": SEED, "image_size": 224},
    "key_format": "site_id|sensor|period_id",
    "versions": {"torch": torch.__version__},
}
(LATD / "manifest.json").write_text(json.dumps(manifest, indent=1))
print(json.dumps(manifest, indent=1))

print("\navailability of the 10 treated sites, P09/P10 (the holdout ground truth):")
t = out_idx.query("group == 'treatment' and period_id in ('before_P09','before_P10')")
print(t.pivot_table(index="site_id", columns=["sensor", "period_id"],
                    values="has_latent", aggfunc="first"))

saved: /data/wang/junh/githubs/latent-synthetic-control/Satellite/notebooks/ts_SCM_ASCM/panel_latent_index.csv (2400, 12)
{
 "date": "2026-08-19",
 "dataset": "biweekly_datasets (60 sites x 2 sensors x 20 periods, 2322 images)",
 "models": {
  "sentinel1": "terramind_v1_tokenizer_s1grd",
  "sentinel2": "terramind_v1_tokenizer_s2l2a"
 },
 "latent": "quantized (encode -> q); flattened for vector arithmetic",
 "shape": {
  "sentinel1": [
   5,
   14,
   14
  ],
  "sentinel2": [
   5,
   14,
   14
  ]
 },
 "preprocessing": "nodata -Inf->NaN->per-band chip mean; bilinear 101->224; TerraMind v1 standardization; S1 VV,VH dB; S2 reflectance x10000, 6-of-12 bands, missing at pretraining mean",
 "decode": {
  "timesteps": 50,
  "seed": 0,
  "image_size": 224
 },
 "key_format": "site_id|sensor|period_id",
 "versions": {
  "torch": "2.13.0+cu130"
 }
}

availability of the 10 treated sites, P09/P10 (the holdout ground truth):
sensor          sentinel1             sentinel2           
period_id     

## Reading

All 2,322 available biweekly images (of 2,400 site-sensor-periods; 78 have no
acquisition) encoded to (5, 14, 14) latents; determinism gate exact (max|diff| = 0 on
re-encode, both sensors); orientation + unit gates pass (chips transposed from the
files' channel-first layout before the verbatim preprocessing). All 10 treated sites
have both sensors at P09 and P10 — the holdout ground truth is complete. Cache:
`data/embeddings_tok_panel/latents_biweekly.npz` (2.1 MB). This npz is 60 sites; the
other 210 of the matching-table 260 were never encoded, which is why notebook 02's kNN
searches 50 controls rather than repeating `embed_DiD`'s 260-control search.